# VoiceOfBank — 07 Classification
**Notebook 7 of 7** — Run on Google Colab (GPU required)

Train a multi-class sentiment classifier on the labelled reviews.

**Task:** Classify a new bank review as Positive / Neutral / Negative

**Two approaches:**
- TF-IDF + XGBoost — classical ML baseline (fast, interpretable)
- Fine-tuned RoBERTa — transformer-based classifier (accurate, production-grade)

**Input:** `data/processed/reviews_bert.csv`

**Output:**
- `data/models/xgb_sentiment.joblib`
- `data/models/roberta_sentiment/` (fine-tuned model)
- `data/processed/classification_results.csv`


## 1. Install Packages

In [ ]:
import subprocess
subprocess.run(
    ['pip', 'install', 'transformers', 'torch', 'datasets',
     'accelerate', 'xgboost', 'joblib', '--quiet'],
    check=True
)
print('Packages ready.')


## 2. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import torch
import joblib
import json
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from torch.utils.data import Dataset
import scipy.sparse

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
RANDOM_SEED = 42
MODEL_NAME = 'cardiffnlp/twitter-roberta-base-sentiment-latest'

print(f'Device : {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')


## 3. Mount Drive and Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PATH  = '/content/drive/MyDrive/VoiceOfBank/data/processed/'
MODELS_PATH = '/content/drive/MyDrive/VoiceOfBank/data/models/'
Path(MODELS_PATH).mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DRIVE_PATH + 'reviews_bert.csv', parse_dates=['date'])

# Use RoBERTa labels as ground truth — more accurate than star ratings
# for borderline cases
# We keep star-rating sentiment as the primary label for consistency
train_df = df[df['split']=='train'].reset_index(drop=True)
test_df  = df[df['split']=='test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}')
print(f'Test : {len(test_df):,}')
print()
print('Label distribution (train):')
print(train_df['sentiment'].value_counts().to_string())


## 4. TF-IDF + XGBoost Baseline

Classical ML pipeline: TF-IDF features + XGBoost classifier.

Fast, interpretable, and strong baseline before fine-tuning.
Uses the cleaned text from notebook 03.


In [ ]:
print('Building TF-IDF matrix...')

tfidf = TfidfVectorizer(
    max_features = 20000,
    ngram_range  = (1, 2),
    min_df       = 5,
    max_df       = 0.95,
    sublinear_tf = True,
)

X_train_tfidf = tfidf.fit_transform(train_df['text_clean'].fillna(''))
X_test_tfidf  = tfidf.transform(test_df['text_clean'].fillna(''))

le = LabelEncoder()
y_train = le.fit_transform(train_df['sentiment'])
y_test  = le.transform(test_df['sentiment'])

print(f'TF-IDF shape : {X_train_tfidf.shape}')
print(f'Classes      : {list(le.classes_)}')
print()

# Hyperparameter tuning via RandomizedSearchCV
# Scoring on Macro F1 — not accuracy — because class imbalance
# makes accuracy misleading (73% of reviews are Positive)
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_space = {
    'n_estimators'     : randint(100, 500),
    'max_depth'        : randint(3, 10),
    'learning_rate'    : uniform(0.01, 0.3),
    'subsample'        : uniform(0.6, 0.4),
    'colsample_bytree' : uniform(0.6, 0.4),
    'min_child_weight' : randint(1, 10),
}

xgb_base = XGBClassifier(
    eval_metric  = 'mlogloss',
    random_state = RANDOM_SEED,
    use_label_encoder = False,
    verbosity    = 0,
    n_jobs       = -1,
)

print('Running RandomizedSearchCV (20 iterations, 3-fold CV)...')
print('Scoring on Macro F1 to handle class imbalance correctly.')
print()

search = RandomizedSearchCV(
    xgb_base,
    param_distributions = param_space,
    n_iter       = 20,
    cv           = 3,
    scoring      = 'f1_macro',
    random_state = RANDOM_SEED,
    n_jobs       = -1,
    verbose      = 1,
)

search.fit(X_train_tfidf, y_train)
xgb = search.best_estimator_

print()
print('Best parameters:')
for param, value in search.best_params_.items():
    print(f'  {param:<25}: {value}')
print(f'\nBest CV Macro F1: {search.best_score_:.4f}')
print()

# Evaluate on test set
y_pred_xgb    = xgb.predict(X_test_tfidf)
y_pred_labels = le.inverse_transform(y_pred_xgb)

print('XGBoost results (tuned):')
print(f'  Accuracy : {accuracy_score(y_test, y_pred_xgb):.4f}')
print(f'  Macro F1 : {f1_score(y_test, y_pred_xgb, average="macro", zero_division=0):.4f}')
print()
print(classification_report(
    test_df['sentiment'], y_pred_labels,
    labels=['Positive','Neutral','Negative'],
    zero_division=0
))


### 4.1 XGBoost Confusion Matrix

In [ ]:
cm = confusion_matrix(
    test_df['sentiment'], y_pred_labels,
    labels=['Positive','Neutral','Negative']
)
cm_df = pd.DataFrame(cm,
    index   = ['True Pos','True Neu','True Neg'],
    columns = ['Pred Pos','Pred Neu','Pred Neg']
)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
            linewidths=0.5, ax=ax)
ax.set_title('XGBoost Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()


### 4.2 Top TF-IDF Features per Class

In [ ]:
# Show which words most strongly predict each sentiment class
feature_names = np.array(tfidf.get_feature_names_out())
class_names   = le.classes_

fig, axes = plt.subplots(1, 3, figsize=(15, 6))

for i, class_name in enumerate(class_names):
    importances = xgb.feature_importances_
    # Use XGBoost's per-class importance via booster
    try:
        booster = xgb.get_booster()
        scores  = booster.get_score(importance_type='gain')
        top_features = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:15]
        words  = [f[0].replace('f','') for f, _ in top_features]
        values = [v for _, v in top_features]
        # Map feature indices to words
        word_labels = [feature_names[int(w)] if w.isdigit() and int(w) < len(feature_names)
                       else w for w in words]
    except Exception:
        top_idx    = xgb.feature_importances_.argsort()[::-1][:15]
        word_labels = feature_names[top_idx]
        values      = xgb.feature_importances_[top_idx]

    color = '#22c55e' if class_name=='Positive' else '#ef4444' if class_name=='Negative' else '#eab308'
    axes[i].barh(range(len(word_labels)), values[:len(word_labels)],
                 color=color, edgecolor='white')
    axes[i].set_yticks(range(len(word_labels)))
    axes[i].set_yticklabels(word_labels[::-1] if len(word_labels) else [], fontsize=8)
    axes[i].set_title(f'{class_name}\nTop Features', fontweight='bold')
    axes[i].set_xlabel('Importance')
    if i == 0: break  # global importance only — per-class not available in XGBoost API

# Use global feature importance instead
top_idx = xgb.feature_importances_.argsort()[::-1][:20]
top_words  = feature_names[top_idx]
top_values = xgb.feature_importances_[top_idx]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(top_words[::-1], top_values[::-1], color='#4f8ef7', edgecolor='white')
ax.set_xlabel('Feature Importance (Gain)')
ax.set_title('Top 20 TF-IDF Features — XGBoost Classifier', fontweight='bold')
sns.despine()
plt.tight_layout()
plt.show()


## 5. Fine-tune RoBERTa

Fine-tune the same RoBERTa model used in notebook 05 on our labelled reviews.

Fine-tuning means we start from the pre-trained weights and train for a few
more epochs on our specific domain data. This adapts the model to the
specific language patterns in UK banking app reviews.

**Training setup:**
- 3 epochs (enough for convergence on 21k training examples)
- Learning rate: 2e-5 (standard for BERT fine-tuning)
- Batch size: 32
- Evaluate on validation set after each epoch


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length     = self.max_len,
            padding        = 'max_length',
            truncation     = True,
            return_tensors = 'pt',
        )
        return {
            'input_ids'     : encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels'        : torch.tensor(self.labels[idx], dtype=torch.long),
        }


print('Loading tokenizer...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Label mapping — consistent with notebook 05
LABEL2ID = {'Negative': 0, 'Neutral': 1, 'Positive': 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# Prepare datasets
# Use a validation split from training data
from sklearn.model_selection import train_test_split
train_texts, val_texts, train_labels, val_labels = train_test_split(
    train_df['text'].tolist(),
    train_df['sentiment'].map(LABEL2ID).tolist(),
    test_size    = 0.10,
    stratify     = train_df['sentiment'],
    random_state = RANDOM_SEED,
)

test_texts  = test_df['text'].tolist()
test_labels = test_df['sentiment'].map(LABEL2ID).tolist()

train_dataset = SentimentDataset(train_texts, train_labels, tokenizer)
val_dataset   = SentimentDataset(val_texts,   val_labels,   tokenizer)
test_dataset  = SentimentDataset(test_texts,  test_labels,  tokenizer)

print(f'Train dataset : {len(train_dataset):,}')
print(f'Val dataset   : {len(val_dataset):,}')
print(f'Test dataset  : {len(test_dataset):,}')


In [ ]:
print('Loading model for fine-tuning...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels = 3,
    id2label   = ID2LABEL,
    label2id   = LABEL2ID,
    ignore_mismatched_sizes = True,
)
model = model.to(DEVICE)
print('Model loaded.')
print(f'Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f} M')


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    acc   = accuracy_score(labels, preds)
    f1    = f1_score(labels, preds, average='macro', zero_division=0)
    return {'accuracy': acc, 'macro_f1': f1}


training_args = TrainingArguments(
    output_dir          = '/content/roberta_checkpoints',
    num_train_epochs    = 3,
    per_device_train_batch_size = 32,
    per_device_eval_batch_size  = 64,
    learning_rate       = 2e-5,
    weight_decay        = 0.01,
    evaluation_strategy = 'epoch',
    save_strategy       = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    logging_steps       = 100,
    seed                = RANDOM_SEED,
    fp16                = True,
    report_to           = 'none',
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    compute_metrics = compute_metrics,
)

print('Starting fine-tuning...')
print(f'Epochs     : 3')
print(f'Batch size : 32')
print(f'Train size : {len(train_dataset):,}')
print()
trainer.train()
print('Fine-tuning complete.')


## 6. Evaluate Fine-tuned RoBERTa

In [ ]:
print('Evaluating on test set...')
predictions = trainer.predict(test_dataset)
y_pred_roberta = predictions.predictions.argmax(axis=1)
y_pred_labels_roberta = [ID2LABEL[p] for p in y_pred_roberta]

acc = accuracy_score(test_labels, y_pred_roberta)
f1  = f1_score(test_labels, y_pred_roberta, average='macro', zero_division=0)

print('Fine-tuned RoBERTa results:')
print(f'  Accuracy : {acc:.4f}')
print(f'  Macro F1 : {f1:.4f}')
print()
print(classification_report(
    test_df['sentiment'], y_pred_labels_roberta,
    labels=['Positive','Neutral','Negative'],
    zero_division=0
))


## 7. Model Comparison

In [ ]:
# Full comparison: VADER vs RoBERTa (zero-shot) vs XGBoost vs Fine-tuned RoBERTa
results = {
    'VADER (zero-shot)' : {
        'acc': accuracy_score(test_df['sentiment'], test_df['vader_label']),
        'f1' : f1_score(test_df['sentiment'], test_df['vader_label'],
                        average='macro', zero_division=0)
    },
    'RoBERTa (zero-shot)' : {
        'acc': accuracy_score(test_df['sentiment'], test_df['bert_label']),
        'f1' : f1_score(test_df['sentiment'], test_df['bert_label'],
                        average='macro', zero_division=0)
    },
    'XGBoost + TF-IDF' : {
        'acc': accuracy_score(y_test, y_pred_xgb),
        'f1' : f1_score(y_test, y_pred_xgb, average='macro', zero_division=0)
    },
    'RoBERTa (fine-tuned)' : {
        'acc': acc,
        'f1' : f1
    },
}

results_df = pd.DataFrame(results).T
print('Full Model Comparison:')
print(results_df.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#9ca3af','#4f8ef7','#f97316','#22c55e']
models = list(results.keys())
x = np.arange(len(models))

axes[0].bar(x, results_df['acc'], color=colors, edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, rotation=20, ha='right', fontsize=9)
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0, 1.05)
axes[0].set_title('Accuracy Comparison', fontweight='bold')
for j, v in enumerate(results_df['acc']):
    axes[0].text(j, v+0.01, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(x, results_df['f1'], color=colors, edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, rotation=20, ha='right', fontsize=9)
axes[1].set_ylabel('Macro F1')
axes[1].set_ylim(0, 1.05)
axes[1].set_title('Macro F1 Comparison', fontweight='bold')
for j, v in enumerate(results_df['f1']):
    axes[1].text(j, v+0.01, f'{v:.3f}', ha='center', fontsize=9)

sns.despine()
plt.suptitle('Sentiment Classification — Full Model Comparison',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


### 7.1 Confusion Matrices — All Models

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 10))
axes = axes.flatten()
labels = ['Positive','Neutral','Negative']

model_preds = [
    ('VADER', test_df['vader_label']),
    ('RoBERTa (zero-shot)', test_df['bert_label']),
    ('XGBoost + TF-IDF', y_pred_labels),
    ('RoBERTa (fine-tuned)', y_pred_labels_roberta),
]

for i, (name, preds) in enumerate(model_preds):
    cm = confusion_matrix(test_df['sentiment'], preds, labels=labels)
    cm_df = pd.DataFrame(cm,
        index   = [f'True {l[:3]}' for l in labels],
        columns = [f'Pred {l[:3]}' for l in labels]
    )
    acc = accuracy_score(test_df['sentiment'], preds)
    sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues',
                linewidths=0.5, ax=axes[i])
    axes[i].set_title(f'{name}\nAccuracy: {acc:.1%}', fontweight='bold')

plt.suptitle('Confusion Matrices — All Models', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 8. Save Models and Results

In [ ]:
# Save XGBoost model and TF-IDF vectorizer
joblib.dump(xgb, MODELS_PATH + 'xgb_sentiment.joblib')
joblib.dump(tfidf, MODELS_PATH + 'tfidf_vectorizer.joblib')
joblib.dump(le, MODELS_PATH + 'label_encoder.joblib')
print('Saved: xgb_sentiment.joblib')
print('Saved: tfidf_vectorizer.joblib')
print('Saved: label_encoder.joblib')

# Save fine-tuned RoBERTa
roberta_path = MODELS_PATH + 'roberta_sentiment/'
trainer.save_model(roberta_path)
tokenizer.save_pretrained(roberta_path)
print(f'Saved: roberta_sentiment/')

# Save classification results
test_df = test_df.copy()
test_df['xgb_pred']     = y_pred_labels
test_df['roberta_pred'] = y_pred_labels_roberta
test_df[[
    'reviewId','bank','text','rating','sentiment',
    'vader_label','bert_label','xgb_pred','roberta_pred'
]].to_csv(DRIVE_PATH + 'classification_results.csv', index=False)
print('Saved: classification_results.csv')

# Save model comparison
results_df.to_csv(DRIVE_PATH + 'model_comparison.csv')
print('Saved: model_comparison.csv')
print()
print('All files saved to Drive.')
print('Download and save to VoiceOfBank/data/processed/ and data/models/')


## 9. Inference Demo

In [ ]:
# Show how to use the fine-tuned model for inference on new reviews
from transformers import pipeline

classifier = pipeline(
    'text-classification',
    model     = model,
    tokenizer = tokenizer,
    device    = 0 if DEVICE.type == 'cuda' else -1,
)

test_reviews = [
    "This app is absolutely brilliant, everything works perfectly",
    "Locked out of my account for 3 days, no help from customer service",
    "App works fine, nothing special about it",
    "Constant crashes after the latest update, absolutely useless",
    "Fast transfers, great notifications, really happy with Monzo",
]

print('Inference demo — fine-tuned RoBERTa:')
print()
for review in test_reviews:
    result = classifier(review, truncation=True, max_length=128)[0]
    label  = result['label']
    score  = result['score']
    print(f'  Text      : {review}')
    print(f'  Predicted : {label} ({score:.1%} confidence)')
    print()
